# 01 — Exploratory Data Analysis (EDA)
**CMD:** `jupyter lab` → open this notebook → **Run All**.

**Purpose:** inspect the original Zhejiang weekly dataset before any modeling, identify data quality concerns, and plot demand and supply.

In [ ]:
# CMD / purpose: Run this cell first in each notebook.
# Makes imports work whether Jupyter starts from the project root or notebooks/.
from pathlib import Path
import os, sys
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open Jupyter from the blood-shortage-ml project directory.')
sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)

### Step 1: Inspect raw dataset
Check file dimensions, variable names, first rows, dates and duplicate entries.

In [ ]:
# CMD / purpose: load the unmodified original CSV and summarize its contents.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
raw = pd.read_csv(ROOT / 'data' / 'Zhejiang_blood_donation.csv')
print('Raw shape:', raw.shape)
display(raw.head())
print('Columns:', list(raw.columns))
print('Exact duplicate rows:', raw.duplicated().sum())

In [ ]:
# CMD / purpose: describe the actual date span and conflicting duplicate dates.
raw_date = pd.to_datetime(raw['date'], format='%m/%d/%Y', errors='coerce')
print('Date span:', raw_date.min().date(), 'to', raw_date.max().date())
print('Repeated date rows:', raw_date.duplicated().sum())
print('String/object columns needing numeric cleanup:')
print(raw.select_dtypes(include=['object']).columns.tolist())

### Step 2: Clean for analysis and visualize the historical patterns
The preprocessing module handles nonbreaking spaces, `#VALUE!`, invalid negative amounts, and duplicate dates. Duplicate-date observations are averaged as an explicit assumption.

In [ ]:
# CMD / purpose: inspect cleaned dates and missing-value counts.
from src.preprocessing import load_and_clean
weekly = load_and_clean()
print('Unique week/date records:', len(weekly))
print(weekly.isna().sum()[weekly.isna().sum()>0].to_string())
display(weekly[['date','demand_total','supply_total','collected_total']].head())

In [ ]:
# CMD / purpose: chart total demand and supply over the final 80 recorded weeks.
ax = weekly.tail(80).plot(x='date', y=['demand_total','supply_total'], figsize=(11,4), title='Historical weekly demand vs supply')
ax.set_ylabel('Units')
plt.tight_layout()
plt.show()

In [ ]:
# CMD / purpose: compare average supply-demand coverage by blood type.
from src.feature_engineering import safe_ratio
coverage = {bt: safe_ratio(weekly[f'supply_{bt}'], weekly[f'demand_{bt}']).mean() * 100 for bt in ['A','B','O','AB']}
ax = pd.Series(coverage).plot(kind='bar', figsize=(7,4), title='Average historical coverage by type (%)')
ax.axhline(80, linestyle='--', color='grey', label='HIGH-risk cutoff')
ax.axhline(100, linestyle=':', color='black', label='LOW-risk cutoff')
ax.set_ylabel('Coverage %'); ax.legend()
plt.tight_layout(); plt.show()

**Interpretation:** Coverage is a project-defined descriptive ratio (supplied/demand), not a validated blood inventory or clinical shortage measure. Historical records are from Zhejiang, not Cambodia.